### ChatInterface + 대화 횟수 세기

- `gr.ChatInterface`로 대화형 채팅 화면
- `history` 매개변수를 실제로 활용해서 "지금까지 몇번 대화했는지" 세는 기능 추가

In [ ]:
import gradio as gr

# ChatInterface에 넘기는 함수는 (message, history) 두개를 받음
# message : 사용자가 방금 입력한 메세지
# history : 지금까지의 대화 기록(현재 메세지는 포함 안됨)

def echo_bot(message, history):
    turn_count = len(history) // 2 + 1
    # history는 [{"role": "user", "content":...},[{"role": "assistant", "content":...}] 로 하나에 두개씩 쌓임
    # 1회 턴이 user + assistant 로 두건 발생됨
    
    return f"[{turn_count}번째 대화] 너가 말한 건: '{message}' 이지?"

In [2]:
demo = gr.ChatInterface(
    fn = echo_bot, 
    title = "대화 횟수를 세는 에코 챗봇",
    description="입력한 말을 그대로 따라 하면서, 지금이 몇 번째 대화인지도 함께 알려줍니다."
)
demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# up pip install openai

In [ ]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import gradio as gr

load_dotenv()
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

def chat_with_gpt(message, history):
    messages= [{"role":h["role"], "content": h["content"]} for h in history]
    messages.append({"role":"user", "content": message})

    response = client.chat.completions.create(
        model = "gpt-4o-mini",
        messages = messages,
    )
    return response.choices[0].message.content

demo.gr.ChatInterface(fn=chat_with_gpt, title="ChatGPT와 대화하기")
demo.launch()


In [1]:
# uv pip install transformers torch

from transformers import pipeline
import gradio as gr

chatbot = pipeline("text-generation", model="microsoft/DialoGPT-medium")

def chat_local(message, history):
    result = chatbot(message, max_length=100)
    return result[0]["generated_text"]

demo = gr.ChatInterface(fn=chat_local, title="로컬 모델 챗봇")
demo.launch()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  863MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

: 

In [4]:
# !pip install google-generativeai
from dotenv import load_dotenv
import os
import google.generativeai as genai
import gradio as gr

load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel("gemini-3.6-flash")   # 빠르고 무료 등급에 넉넉한 모델

def chat_with_gemini(message, history):
    # Gemini는 role 이름이 다름: assistant -> model
    contents = [
        {"role": "user" if h["role"] == "user" else "model",
         "parts": [h["content"]]}
        for h in history
    ]
    contents.append({"role": "user", "parts": [message]})

    response = model.generate_content(contents)
    return response.text

demo = gr.ChatInterface(fn=chat_with_gemini, title="Gemini와 대화하기")
demo.launch()


* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
